## 결과 확인

이번에는 사전학습 모델(Vanilla)과 직접 학습시킨 두 모델을 비교하는 실습을 진행합니다.

**Vanilla(Pretrained)**  
`사전 학습된 LLaMA-3-1B Instruct 모델을 사용합니다. 다양한 질문과 지시에 대해 기본적인 성능을 보여줍니다.`

**Supervised Fine-Tuning(SFT)**  
`02_model_training_SFT.ipynb에서 모델에게 정답을 강제로 보여주면서 학습한 모델입니다.`

**Direct Preference Optimization(DPO)**  
`03_model_training_DPO.ipynb에서 사용자 선호도를 반영한 pairwise 학습으로 인간처럼 더 선호되는 출력을 생성하는 모델입니다.`

SFT와 DPO 데이터셋이 별도로 구축되어 스타일이 달라 정량적 평가는 의미가 없다고 판단됩니다.  
따라서 각 모델의 출력 결과를 직접 확인하는 정성적 비교만 진행하겠습니다.

In [ ]:
import os

# 미리 저장된 모델을 사용하기 위해 환경변수 설정
os.environ["HF_HUB_CACHE"] = "/mnt/elice/dataset/hub"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import gc
from dotenv import load_dotenv  # 환경변수 로딩
load_dotenv("env.txt")

# 공통 설정
max_new_tokens = 300
prompt_template = """<|start_header_id|>system<|end_header_id|>
{}<|eot_id|><|start_header_id|>user<|end_header_id|>
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

# 테스트용 system + user 프롬프트
system_prompt = "주어진 정보를 바탕으로 질문에 순서대로 답하세요."
user_prompt = """
정보:
대구 FC는 2002년 FIFA 월드컵의 축구 붐에 힘입어 대한민국 최초로 시민 구단의 개념을 도입하면서 창단되었다. 월드컵이 끝난 후 2002년 8월 6일 창립 회의를 시작으로 2002년 10월 9일 (주)대구시민프로축구단이라는 공식 명칭으로 첫 창립 총회를 열면서 창단하게 된다. 이후 2002년 11월 15일부터 12월 24일간 1차 시민주 공모를 실시해 127억원의 자금을 확보하였으며, 2002년 12월 26일 한국프로축구연맹에 의해 창단이 승인되었다. 2003년 3월 19일 공식 창단식을 거행하였으며 초대 감독으로 1980년대와 1990년대 대한민국 국가대표팀을 이끌었던 박종환을 선임하고 K리그 2003 시즌에 참가하였다.

질문:
1. 대구 FC는 몇 년에 창단되었나요?
2. 초대 감독은 누구였나요?
3. 첫 시즌에 확보한 자금은 얼마였나요?"""

full_prompt = prompt_template.format(system_prompt, user_prompt)

# 추론 함수 정의
def generate_response(model, tokenizer, prompt):
    # 입력 프롬프트를 토크나이저로 토큰화, 텐서 변환 후 모델이 실행 중인 장치(CPU 또는 GPU)로 이동
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 모델에 입력을 전달하여 출력 생성
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, # 생성할 최대 새 토큰 수
            do_sample=True,
            temperature=0.7,
            top_p=0.9, # 누적 확률 기반 샘플링 (상위 90% 확률의 토큰 중 선택, 다양성 조절)
            pad_token_id=tokenizer.eos_token_id # 패딩 토큰 ID를 EOS(문장 종료) 토큰으로 설정
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 메모리 정리 함수 정의
def clear_memory(model):
	gc.collect()
	torch.cuda.empty_cache()
	del model
	# CUDA context 삭제 (단, 새 모델 로드시 다시 생성됨)
	for obj in gc.get_objects():
		try:
			if torch.is_tensor(obj):
				del obj
		except:
			pass

### 1. Vanilla 모델

In [ ]:
# 모델 불러오기
vanilla_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", torch_dtype=torch.float16, device_map="auto")
vanilla_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

In [ ]:
print("\n[Vanilla Model 출력]")
print(generate_response(vanilla_model, vanilla_tokenizer, full_prompt))

### 2. SFT 모델

In [ ]:
# Step 1. SFT 모델의 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained("final_adapter")

# Step 2. base 모델의 embedding 크기 tokenizer에 맞게 맞추기
vanilla_model.resize_token_embeddings(len(tokenizer))

# Step 3. 학습된 LoRA adapter 적용
sft_model = PeftModel.from_pretrained(vanilla_model, "final_adapter")  # 기본 모델 및 학습한 LoRA adapter 경로

In [ ]:
print(generate_response(sft_model, tokenizer, full_prompt))

### 3. DPO 모델

In [ ]:
clear_memory(sft_model)
# Step 4. 학습된 DPO LoRA adapter 적용
dpo_model = PeftModel.from_pretrained(
    vanilla_model,
    "dpo_model/checkpoint-10556",  # 학습한 LoRA adapter 경로
)

In [ ]:
print(generate_response(dpo_model, tokenizer, full_prompt))

### 고찰
이번 실습에서 Vanilla, SFT, DPO 모델에 다양한 입력을 제공하여 출력 결과를 비교했습니다.
아래 항목들을 통해 실습 결과를 심층적으로 분석하고, 모델의 동작과 학습 과정에 대해 고찰해봅시다.

1. 이번 실습에서는 학습 데이터에 대한 탐색적 데이터 분석(EDA)을 깊이 다루지 않았습니다.  
학습 데이터셋의 특성(예: 데이터 크기, 주제, 형식, 편향)을 간단히 살펴보고,  
각 모델(Vanilla, SFT, DPO)의 출력 결과와 데이터셋의 특성이 어떻게 연관되는지 생각해봅시다.

2. 동일한 입력에 대해 각 모델이 얼마나 일관된 응답을 생성하는지, 또는 얼마나 다양한 응답을 제공하는지 분석해봅시다. 

3. 결과가 생각보다 나쁠 경우, 이후에 어떤 조치를 취할 수 있을지 논의해봅시다.